# 公司政策智能问答系统（RAG）

流程：
1. 加载文档（TXT / PDF，按页保留来源信息）
2. 数据分块
3. 文本嵌入模型
4. 构建向量数据库
5. 结合 LLM 回答问题，并附上出处（文件名 + 页码）
6. 检索效果评估（Hit@k），对比不同分块大小

依赖：`pip install langchain-huggingface langchain-ollama langchain-chroma pymupdf`

目录结构：PDF / TXT 文件放进 `knowledge_base` 文件夹，`eval_set.json` 和本 notebook 放在同一目录下。

In [ ]:
# ===== 配置 =====
from pathlib import Path

KB_DIR = Path("./knowledge_base")   # 知识库文件夹，里面的 PDF / TXT 会全部被加载
DOC_FILES = sorted(list(KB_DIR.glob("*.pdf")) + list(KB_DIR.glob("*.txt")))
EVAL_FILE = "./eval_set.json"
CHUNK_SIZE = 200                 # 分块对比实验中 200 的 Hit@1 最高
CHUNK_OVERLAP = 50               # 适当重叠，避免关键信息落在两个块的边界上
TOP_K = 3
PERSIST_DIR = "./chroma_db"
EMBED_MODEL = "BAAI/bge-small-zh-v1.5"
LLM_MODEL = "qwen2.5:7b"


In [ ]:
# ===== 1. 加载文档 =====
import re
import pymupdf   # 旧写法 import fitz 已过时
from langchain_core.documents import Document

CJK = r"\u4e00-\u9fff\u3000-\u303f\uff00-\uffef"   # 中文字符和中文标点


def clean_text(text):
    """清理 PDF 提取出的文本：
    1. PDF 每行末尾都有换行：中文之间的换行直接去掉，其他换行替换为空格（避免英文单词粘连）
    2. 去掉中文与数字/字母之间多余的空格（例如“600 元”→“600元”）"""
    text = re.sub(rf"(?<=[{CJK}])\n(?=[{CJK}])", "", text)
    text = text.replace("\n", " ")
    text = re.sub(rf"(?<=[{CJK}])\s+(?=[0-9A-Za-z%])|(?<=[0-9A-Za-z%])\s+(?=[{CJK}])", "", text)
    text = re.sub(rf"(?<=[{CJK}])\s+(?=[{CJK}])", "", text)
    return re.sub(r" {2,}", " ", text).strip()


def load_pdf(path, skip_pages=0):
    """按页加载 PDF：先合并整页文本，再在“第X章 / 第X条”前插入段落分隔，
    让分块优先在条款边界切分。metadata 中记录文件名和页码（从 1 开始）。"""
    path = Path(path)
    docs = []
    with pymupdf.open(path) as pdf:
        for i, page in enumerate(pdf):
            if i < skip_pages:
                continue
            text = clean_text(page.get_text("text"))
            text = re.sub(r"\s*(第[一二三四五六七八九十百]+[章条])", r"\n\n\1", text).strip()
            if text:
                docs.append(Document(page_content=text,
                                     metadata={"source": path.name, "page": i + 1}))
    return docs


def load_txt(path):
    path = Path(path)
    text = path.read_text(encoding="utf-8")
    return [Document(page_content=text, metadata={"source": path.name, "page": None})]


def load_documents(files):
    docs = []
    for f in files:
        if f.suffix.lower() == ".pdf":
            docs.extend(load_pdf(f))
        elif f.suffix.lower() == ".txt":
            docs.extend(load_txt(f))
    return docs


documents = load_documents(DOC_FILES)
print(f"共加载 {len(DOC_FILES)} 个文件，{len(documents)} 页：")
for f in DOC_FILES:
    print("  -", f.name)

In [ ]:
# ===== 2. 数据分块 =====
from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        # 先按段落切，再按中文句子切
        separators=["\n\n", "\n", "。", "！", "？", "；", " ", ""],
    )
    return splitter.split_documents(docs)   # 分块后会保留原 Document 的 metadata（来源、页码）


splits = split_documents(documents)
print(f"共切分为 {len(splits)} 个文本块")

In [ ]:
# ===== 3. 文本嵌入模型 =====
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    cache_folder="./bge-small-zh-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},   # 归一化后用余弦相似度更准确
)

In [ ]:
# ===== 4. 构建向量数据库 =====
import shutil

try:
    from langchain_chroma import Chroma                     # 推荐的新包
except ImportError:
    from langchain_community.vectorstores import Chroma     # 兼容旧环境

# 重复运行 from_documents 会把同样的数据再写入一遍，导致检索结果重复，所以先清空旧库
shutil.rmtree(PERSIST_DIR, ignore_errors=True)

db = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
)
retriever = db.as_retriever(search_kwargs={"k": TOP_K})

In [ ]:
# ===== 5. 结合 LLM 回答问题（附出处） =====
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat = ChatOllama(model=LLM_MODEL, temperature=0)   # temperature=0 让回答更稳定

prompt = ChatPromptTemplate.from_template(
    "你是公司政策问答助手。请只根据下面的上下文回答问题，并在回答末尾用 [编号] 标注依据。\n"
    "如果上下文中没有答案，就回答“根据现有文档无法回答”，不要编造。\n\n"
    "上下文：\n{context}\n\n"
    "问题：{question}"
)
generate_chain = prompt | chat | StrOutputParser()


def source_label(doc):
    src = doc.metadata.get("source", "未知来源")
    page = doc.metadata.get("page")
    return f"{src} 第{page}页" if page else src


def format_docs(docs):
    return "\n\n".join(
        f"[{i}] ({source_label(d)})\n{d.page_content}" for i, d in enumerate(docs, 1)
    )


def answer_question(question):
    # 先检索一次，再把同一批文档用于生成回答和展示出处
    docs = retriever.invoke(question)
    answer = generate_chain.invoke({"context": format_docs(docs), "question": question})
    sources = [{"label": source_label(d), "content": d.page_content} for d in docs]
    return {"answer": answer, "sources": sources}

In [ ]:
# ===== 测试 =====
test_questions = [
    "年假的有效期是多久？",
    "去深圳出差，住酒店每晚最多能报多少钱？",
    "公司提供免费午餐吗？",    # 后两个问题文档里没有答案，用来检查模型会不会编造
    "公司有没有健身房？",
]
for q in test_questions:
    result = answer_question(q)
    print("问题：", q)
    print("回答：", result["answer"])
    print("出处：")
    for i, s in enumerate(result["sources"], 1):
        print(f"  [{i}] {s['label']}：{s['content'][:60]}...")
    print("-" * 50)

## 6. 检索效果评估

**Hit@k**：对每个问题检索前 k 个文本块，只要其中有一个块**来自正确的文件**并且**包含标准答案的关键词**，就算命中。命中的问题数 ÷ 总问题数 = Hit@k。

同时检查来源，是因为以后加入法律法规文件后，同一个关键词（比如年假天数）可能出现在多个文件里。

评估集 `eval_set.json` 中每条包含 `question`（问题）、`keyword`（答案关键词）、`source`（答案所在文件），可以继续往里添加问题。

In [ ]:
# ===== 加载评估集 =====
import json

with open(EVAL_FILE, encoding="utf-8") as f:
    eval_set = json.load(f)
print(f"评估集共 {len(eval_set)} 个问题")

In [ ]:
# ===== Hit@k 计算 =====
def normalize(text):
    return re.sub(r"\s+", "", text)   # 去掉所有空白字符，避免换行或空格影响关键词匹配


def hit_at_k(vectorstore, eval_set, k=TOP_K):
    hits, misses = 0, []
    for item in eval_set:
        docs = vectorstore.similarity_search(item["question"], k=k)
        keyword = normalize(item["keyword"])
        if any(d.metadata.get("source") == item["source"] and keyword in normalize(d.page_content)
               for d in docs):
            hits += 1
        else:
            misses.append(item["question"])
    return hits / len(eval_set), misses


score, misses = hit_at_k(db, eval_set)
print(f"当前配置 chunk_size={CHUNK_SIZE}：Hit@{TOP_K} = {score:.1%}")
if misses:
    print("未命中的问题：")
    for q in misses:
        print("  -", q)

In [ ]:
# ===== 对比不同分块大小 =====
results = []
for size in [50, 100, 200, 400, 800]:
    overlap = min(CHUNK_OVERLAP, size // 4)
    chunks = split_documents(documents, chunk_size=size, chunk_overlap=overlap)
    # 用内存中的临时集合做实验，不影响上面持久化的数据库
    tmp_db = Chroma.from_documents(chunks, embeddings, collection_name=f"eval_chunk_{size}")
    score1, _ = hit_at_k(tmp_db, eval_set, k=1)
    score3, _ = hit_at_k(tmp_db, eval_set, k=3)
    results.append((size, overlap, len(chunks), score1, score3))
    tmp_db.delete_collection()   # 两个分数都算完后再删除

print(f"{'chunk_size':>10} {'overlap':>8} {'块数':>6} {'Hit@1':>8} {'Hit@3':>8}")
for size, overlap, n, s1, s3 in results:
    print(f"{size:>10} {overlap:>8} {n:>6} {s1:>8.1%} {s3:>8.1%}")